In [1]:
tabla='ctsci10'
col_essi='fec_sol'
col_tabla='solcitafec'
essi='essi_dat_cex003_etl'
col_dat='fec_sol'
dat='dat_cext003_essi'

In [2]:
from decouple import config
import decouple #eliminar en .py
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta
import time 
from datetime import datetime
from sqlalchemy import text
import oracledb
import sys

In [3]:
inicioTiempo = time.time()
inicioProceso=time.time()
now_inicio = datetime.now()

In [4]:
######################FUNCIONES DE LOG###########################
global dim, control_a, control_d, base1, falla, merge

control_a=[]
control_d=[]
dim=[]
falla=[]
id_afectado=[]

def log_falla(id, cod, isNeeded):
    if isNeeded:
        filas_perdidas = merge.loc[pd.isnull(merge[id])]
        filas_perdidas = filas_perdidas.drop_duplicates(subset=[cod])
        filas_perdidas=filas_perdidas[[cod]]
        if filas_perdidas.empty:
            filas_perdidas_string = 0
        else:
            filas_perdidas_string = filas_perdidas.to_string(index=False, header=False)
            filas_perdidas_string = filas_perdidas_string.replace('\n', ',')
    else:
        filas_perdidas_string = 0
    falla.append(filas_perdidas_string)
    id_afectado.append(id)

def log_control(table):    
    dim.append(table)
    control_d.append(base1.shape[0])
    control_a.append(base1.shape[0])

In [5]:
config = decouple.AutoConfig(' ')
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="essi_etl"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="etl_logs"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

In [6]:
#connection1.close()
#connection2.close()
#connection3.close()

In [7]:
from datetime import datetime
from sqlalchemy import text


fecha = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=3", con=connection2)
fecha= fecha.iloc[0, 0]
print(fecha)

now = datetime.now()

query=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_mod=3"

c1= text(query)
connection2.execute(c1)

2023-07-01


In [8]:
fecha='2023-06-01'

In [9]:
base1=pd.read_sql_query(f"SELECT * FROM {essi} WHERE {col_essi} >='{fecha}'", con=connection1)

In [10]:
base1.columns

Index(['num_sol', 'fec_sol', 'ori_cas', 'cod_cas', 'des_cas', 'cod_red',
       'des_red', 'pac_sec', 'cod_are', 'des_are', 'cod_ser', 'des_ser',
       'cod_act', 'des_act', 'cod_sub', 'des_sub', 'cod_tci', 'des_tci',
       'fec_pre', 'cod_dpc', 'des_dpc', 'cod_tpc', 'des_tpc', 'cod_est',
       'des_est', 'ori_sol', 'cas_sol', 'des_cso', 'act_med', 'est_reg',
       'des_reg', 'usu_cre', 'fec_cre', 'usu_mod', 'fec_mod', 'cod_pri',
       'des_pri', 'ip_cre', 'usu_anu', 'fec_anu', 'ip_mod', 'mod_cre',
       'mod_anu', 'mod_mod', 'sec_num', 'cod_cps', 'med_fis', 'tip_ter',
       'ori_sol_ori', 'cas_sol_ori', 'act_med_ori', 'ip_anu'],
      dtype='object')

In [11]:
base1.shape

(144167, 52)

In [12]:
base2=pd.read_sql_query(f"SELECT * FROM {dat} LIMIT 10", con=connection2)

In [13]:
base1["fec_cre"]

0        2023-06-02 14:36:41+00:00
1        2023-06-02 13:52:15+00:00
2        2023-06-02 15:56:03+00:00
3        2023-06-02 15:25:26+00:00
4        2023-06-02 15:35:16+00:00
                    ...           
144162   2023-06-01 19:37:20+00:00
144163   2023-06-01 22:08:13+00:00
144164   2023-06-01 22:54:49+00:00
144165   2023-06-01 19:37:32+00:00
144166   2023-06-02 00:05:01+00:00
Name: fec_cre, Length: 144167, dtype: datetime64[ns, UTC]

In [14]:
tiempo = pd.read_sql_query(f"SELECT id_tiempo,dt_fecha,dt_mes,dt_dia,dt_dia_sem,dt_sem,dt_ano FROM dim_tiempo", con=connection2)
tiempo=tiempo.rename(columns={"id_tiempo":"id_time_sol","dt_fecha":"fec_sol","dt_mes":"mes_sol","dt_dia":"dia_sol","dt_dia_sem":"dia_sem_sol","dt_sem":"sem_sol","dt_ano":"ano_sol"})
base1=pd.merge(base1,tiempo,how='left',on='fec_sol')


tiempo = pd.read_sql_query(f"SELECT id_tiempo,dt_fecha,dt_mes,dt_dia,dt_dia_sem,dt_sem,dt_ano FROM dim_tiempo", con=connection2)
tiempo=tiempo.rename(columns={"id_tiempo":"id_time_pre","dt_fecha":"fec_pre","dt_mes":"mes_pre","dt_dia":"dia_pre","dt_dia_sem":"dia_sem_pre","dt_sem":"sem_pre","dt_ano":"ano_pre"})
base1=pd.merge(base1,tiempo,how='left',on='fec_pre')


base1['fec_cre_temp']=base1['fec_cre']
base1['fec_mod_temp']=base1['fec_mod']
base1['fec_anu_temp']=base1['fec_anu']

tiempo = pd.read_sql_query(f"SELECT id_tiempo,dt_fecha,dt_mes,dt_dia,dt_dia_sem,dt_sem,dt_ano FROM dim_tiempo", con=connection2)
tiempo=tiempo.rename(columns={"id_tiempo":"id_time_cre","dt_fecha":"fec_cre","dt_mes":"mes_cre","dt_dia":"dia_cre","dt_dia_sem":"dia_sem_cre","dt_sem":"sem_cre","dt_ano":"ano_cre"})
base1['fec_cre'] = pd.to_datetime(base1['fec_cre']).dt.date
base1=pd.merge(base1,tiempo,how='left',on='fec_cre')


tiempo = pd.read_sql_query(f"SELECT id_tiempo,dt_fecha,dt_mes,dt_dia,dt_dia_sem,dt_sem,dt_ano FROM dim_tiempo", con=connection2)
tiempo=tiempo.rename(columns={"id_tiempo":"id_time_mod","dt_fecha":"fec_mod","dt_mes":"mes_mod","dt_dia":"dia_mod","dt_dia_sem":"dia_sem_mod","dt_sem":"sem_mod","dt_ano":"ano_mod"})
base1['fec_mod'] = pd.to_datetime(base1['fec_mod']).dt.date
base1=pd.merge(base1,tiempo,how='left',on='fec_mod')


tiempo = pd.read_sql_query(f"SELECT id_tiempo,dt_fecha,dt_mes,dt_dia,dt_dia_sem,dt_sem,dt_ano FROM dim_tiempo", con=connection2)
tiempo=tiempo.rename(columns={"id_tiempo":"id_time_anu","dt_fecha":"fec_anu","dt_mes":"mes_anu","dt_dia":"dia_anu","dt_dia_sem":"dia_sem_anu","dt_sem":"sem_anu","dt_ano":"ano_anu"})
base1['fec_anu'] = pd.to_datetime(base1['fec_anu']).dt.date
base1=pd.merge(base1,tiempo,how='left',on='fec_anu')


base1['fec_cre'] = base1['fec_cre_temp']
base1['fec_mod'] = base1['fec_mod_temp']
base1['fec_anu'] = base1['fec_anu_temp']

base1 = base1.drop('fec_cre_temp', axis=1)
base1 = base1.drop('fec_mod_temp', axis=1)
base1 = base1.drop('fec_anu_temp', axis=1)

In [15]:
control_a.append(base1.shape[0])

In [16]:
oricas = pd.read_sql_query(f"SELECT id_oricas,ori_cod FROM dim_oricas", con=connection2)
oricas=oricas.rename(columns={"ori_cod":"ori_cas"})
merge=pd.merge(base1,oricas,how='left',on='ori_cas')
base1=pd.merge(base1,oricas,how='inner',on='ori_cas')
base1.shape

(144167, 83)

In [17]:
log_falla('id_oricas', 'ori_cas', True)
log_control('dim_oricas')
base1=base1.drop('ori_cas',axis=1)

In [18]:
cas = pd.read_sql_query(f"SELECT id_cas,cod_cas FROM dim_cas where id_red is not null", con=connection2)
cas = cas.dropna()
merge=pd.merge(base1,cas,how='left',on='cod_cas')
base1=pd.merge(base1,cas,how='inner',on='cod_cas')
base1.shape

(144167, 83)

In [19]:
log_falla('id_cas', 'cod_cas', True)
log_control('dim_cas')
base1=base1.drop('cod_cas',axis=1)

In [20]:
red = pd.read_sql_query(f"SELECT id_red,cod_red FROM dim_red", con=connection2)
merge=pd.merge(base1,red,how='left',on='cod_red')
base1=pd.merge(base1,red,how='inner',on='cod_red')
base1.shape

(144167, 83)

In [21]:
log_falla('id_red', 'cod_red', True)
log_control('dim_red')
base1=base1.drop('cod_red',axis=1)

In [22]:
base1 = base1.rename(columns={'pac_sec':'id_pacsec'})
base1.shape

(144167, 82)

In [23]:
are = pd.read_sql_query(f"SELECT id_area,cod_are FROM dim_areas", con=connection2)
merge=pd.merge(base1,are,how='left',on='cod_are')
base1=pd.merge(base1,are,how='inner',on='cod_are')
base1.shape

(144167, 83)

In [24]:
log_falla('id_area', 'cod_are', True)
log_control('dim_areas')
base1=base1.drop('cod_are',axis=1)

In [25]:
serv= pd.read_sql_query(f"SELECT id_serv,cod_ser FROM dim_servicios", con=connection2)
merge=pd.merge(base1,serv,how='left',on='cod_ser')
base1=pd.merge(base1,serv,how='inner',on='cod_ser')
base1.shape

(144167, 83)

In [26]:
log_falla('id_serv', 'cod_ser', True)
log_control('dim_servicios')
base1=base1.drop('cod_ser',axis=1)

In [27]:
activi = pd.read_sql_query(f"SELECT id_activi,cod_act FROM dim_activi", con=connection2)
merge=pd.merge(base1,activi,how='left',on='cod_act')
base1=pd.merge(base1,activi,how='inner',on='cod_act')
base1.shape

(144167, 83)

In [28]:
log_falla('id_activi', 'cod_act', True)
log_control('dim_activi')

In [29]:
subacti = pd.read_sql_query(f"SELECT id_subacti,cod_sub,cod_act FROM dim_subacti", con=connection2)
subacti["KEY"]=subacti["cod_sub"]+subacti["cod_act"]
subacti=subacti.drop(["cod_sub",'cod_act'], axis=1)
base1["KEY"]=base1["cod_sub"].astype(str)+base1['cod_act'].astype(str)
base1["KEY"]=base1["KEY"].str.replace(' ', '', regex=True)
subacti["KEY"]=subacti["KEY"].str.replace(' ', '', regex=True)
merge = pd.merge(base1,subacti,how='left',on='KEY')
base1 = pd.merge(base1,subacti,how='inner',on='KEY')

base1.shape

(144167, 85)

In [30]:
log_falla('id_subacti', 'KEY', True)
log_control('dim_subacti')
base1=base1.drop('KEY', axis=1)
base1=base1.drop('cod_act', axis=1)

In [31]:
tipcit = pd.read_sql_query(f"SELECT id_tipcit,cod_tci FROM dim_tipcit", con=connection2)
tipcit = tipcit.rename(columns={'id_tipcit':'id_tci'})
merge=pd.merge(base1,tipcit,how='left',on='cod_tci')
base1=pd.merge(base1,tipcit,how='inner',on='cod_tci')
base1.shape

(144167, 84)

In [32]:
log_falla('id_tci', 'cod_tci', True)
log_control('dim_tipcit')
base1=base1.drop('cod_tci', axis=1)

In [33]:
diapresol = pd.read_sql_query(f"SELECT id_diapre,cod_dps FROM dim_cexdiapresol", con=connection2)
diapresol = diapresol.rename(columns={'cod_dps':'cod_dpc'})
diapresol = diapresol.rename(columns={'id_diapre':'id_dpc'})
merge=pd.merge(base1,diapresol,how='left',on='cod_dpc')
base1=pd.merge(base1,diapresol,how='inner',on='cod_dpc')
base1.shape

(144167, 84)

In [34]:
log_falla('id_dpc', 'cod_dpc', True)
log_control('dim_cexdiapresol')
base1=base1.drop('cod_dpc', axis=1)

In [35]:
turprecit = pd.read_sql_query(f"SELECT id_turpre,cod_tpc FROM dim_cexturprecit", con=connection2)
turprecit = turprecit.rename(columns={'id_turpre':'id_tpc'})
merge=pd.merge(base1,turprecit,how='left',on='cod_tpc')
base1=pd.merge(base1,turprecit,how='inner',on='cod_tpc')
base1.shape

(144167, 84)

In [36]:
log_falla('id_tpc', 'cod_tpc', True)
log_control('dim_cexturprecit')
base1=base1.drop('cod_tpc', axis=1)

In [37]:
estsolcit = pd.read_sql_query(f"SELECT id_estsolcit,cod_esc FROM dim_cexestsolcit", con=connection2)
estsolcit = estsolcit.rename(columns={'id_estsolcit':'id_estado'})
estsolcit = estsolcit.rename(columns={'cod_esc':'cod_est'})
merge=pd.merge(base1,estsolcit,how='left',on='cod_est')
base1=pd.merge(base1,estsolcit,how='inner',on='cod_est')
base1.shape

(144167, 84)

In [38]:
log_falla('id_estado', 'cod_est', True)
log_control('dim_cexestsolcit')
base1=base1.drop('cod_est', axis=1)

In [39]:
orisol = pd.read_sql_query(f"SELECT id_oricas,ori_cod FROM dim_oricas", con=connection2)
orisol=orisol.rename(columns={"id_oricas":"id_orisol"})
orisol=orisol.rename(columns={"ori_cod":"ori_sol"})
merge=pd.merge(base1,orisol,how='left',on='ori_sol')
base1=pd.merge(base1,orisol,how='left',on='ori_sol')
base1.shape

(144167, 84)

In [40]:
log_falla('id_orisol', 'ori_sol', True)
log_control('dim_oricas')
base1=base1.drop('ori_sol',axis=1)

In [41]:
cassol = pd.read_sql_query(f"SELECT id_cas,cod_cas FROM dim_cas where id_red is not null", con=connection2)
cassol = cassol.dropna()
cassol=cassol.rename(columns={"id_cas":"id_cassol"})
cassol=cassol.rename(columns={"cod_cas":"cas_sol"})
merge=pd.merge(base1,cassol,how='left',on='cas_sol')
base1=pd.merge(base1,cassol,how='left',on='cas_sol')
base1.shape

(144167, 84)

In [42]:
log_falla('id_cassol', 'cas_sol', True)
log_control('dim_cas')
base1=base1.drop('cas_sol',axis=1)

In [43]:
estreg = pd.read_sql_query(f"SELECT id_estreg,cod_reg FROM dim_cexestreg", con=connection2)
estreg = estreg.rename(columns={"cod_reg":"est_reg"})
merge=pd.merge(base1,estreg,how='left',on='est_reg')
base1=pd.merge(base1,estreg,how='inner',on='est_reg')
base1.shape

(144167, 84)

In [44]:
log_falla('id_estreg', 'est_reg', True)
log_control('dim_cexestreg')
base1=base1.drop('est_reg', axis=1)

In [45]:
numdoc = pd.read_sql_query(f"SELECT id_person, num_doc FROM dim_personal", con=connection2)
numdoc=numdoc.drop_duplicates(subset="num_doc")
numdoc=numdoc.rename(columns={"num_doc":"usu_cre"})
numdoc=numdoc.rename(columns={"id_person":"id_usucre"})
merge=pd.merge(base1,numdoc,how='left',on='usu_cre')
base1=pd.merge(base1,numdoc,how='left',on='usu_cre')
base1.shape

(144167, 84)

In [46]:
log_falla('id_usucre', 'usu_cre', True)
log_control('dim_personal')
base1=base1.drop('usu_cre',axis=1)

In [47]:
numdoc = pd.read_sql_query(f"SELECT id_person, num_doc FROM dim_personal", con=connection2)
numdoc=numdoc.drop_duplicates(subset="num_doc")
numdoc=numdoc.rename(columns={"num_doc":"usu_mod"})
numdoc=numdoc.rename(columns={"id_person":"id_usumod"})
merge=pd.merge(base1,numdoc,how='left',on='usu_mod')
base1=pd.merge(base1,numdoc,how='left',on='usu_mod')
base1.shape

(144167, 84)

In [48]:
log_falla('id_usumod', 'usu_mod', True)
log_control('dim_personal')
base1=base1.drop('usu_mod',axis=1)

In [49]:
numdoc = pd.read_sql_query(f"SELECT id_person, num_doc FROM dim_personal", con=connection2)
numdoc=numdoc.drop_duplicates(subset="num_doc")
numdoc=numdoc.rename(columns={"num_doc":"usu_anu"})
numdoc=numdoc.rename(columns={"id_person":"id_asuanu"})
merge=pd.merge(base1,numdoc,how='left',on='usu_anu')
base1=pd.merge(base1,numdoc,how='left',on='usu_anu')
base1.shape

(144167, 84)

In [50]:
log_falla('id_asuanu', 'usu_anu', True)
log_control('dim_personal')
base1=base1.drop('usu_anu',axis=1)

In [51]:
prisol = pd.read_sql_query(f"SELECT id_prisol,cod_pri FROM dim_cexprisol", con=connection2)
prisol=prisol.rename(columns={"id_prisol":"id_priori"})
prisol['cod_pri'] = prisol['cod_pri'].astype(int)
merge=pd.merge(base1,prisol,how='left',on='cod_pri')
base1=pd.merge(base1,prisol,how='inner',on='cod_pri')
base1.shape

(144167, 84)

In [52]:
log_falla('id_priori', 'cod_pri', True)
log_control('dim_cexprisol')
base1=base1.drop('cod_pri', axis=1)

In [53]:
base1=base1.rename(columns={"mod_cre":"id_crea"})
base1=base1.rename(columns={"mod_anu":"id_anula"})
base1=base1.rename(columns={"mod_mod":"id_modif"})

In [54]:
cps = pd.read_sql_query(f"SELECT id_cps,cod_cps FROM dim_cps", con=connection2)
cps=cps.rename(columns={"id_prisol":"id_priori"})
merge=pd.merge(base1,cps,how='left',on='cod_cps')
base1=pd.merge(base1,cps,how='left',on='cod_cps')
base1.shape

(144167, 84)

In [55]:
log_falla('id_cps', 'cod_cps', True)
base1=base1.drop('cod_cps',axis=1)
dim.append('dim_cps')
control_d.append(base1.shape[0])

In [56]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df2_columns - df1_columns
different_columns

set()

In [57]:
borrando=f"DELETE FROM {dat} WHERE {col_dat} >='{fecha}'"
borrado = connection2.execute(borrando)

In [58]:
query_count_before = f"SELECT COUNT(*) FROM {dat}"
cant_antes = connection2.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {dat} antes de la inserción: {cant_antes}")

Cantidad de filas en la tabla dat_cext003_essi antes de la inserción: 2210143


In [59]:
base1.columns

Index(['num_sol', 'fec_sol', 'des_cas', 'des_red', 'id_pacsec', 'des_are',
       'des_ser', 'des_act', 'cod_sub', 'des_sub', 'des_tci', 'fec_pre',
       'des_dpc', 'des_tpc', 'des_est', 'des_cso', 'act_med', 'des_reg',
       'fec_cre', 'fec_mod', 'des_pri', 'ip_cre', 'fec_anu', 'ip_mod',
       'id_crea', 'id_anula', 'id_modif', 'sec_num', 'med_fis', 'tip_ter',
       'ori_sol_ori', 'cas_sol_ori', 'act_med_ori', 'ip_anu', 'id_time_sol',
       'mes_sol', 'dia_sol', 'dia_sem_sol', 'sem_sol', 'ano_sol',
       'id_time_pre', 'mes_pre', 'dia_pre', 'dia_sem_pre', 'sem_pre',
       'ano_pre', 'id_time_cre', 'mes_cre', 'dia_cre', 'dia_sem_cre',
       'sem_cre', 'ano_cre', 'id_time_mod', 'mes_mod', 'dia_mod',
       'dia_sem_mod', 'sem_mod', 'ano_mod', 'id_time_anu', 'mes_anu',
       'dia_anu', 'dia_sem_anu', 'sem_anu', 'ano_anu', 'id_oricas', 'id_cas',
       'id_red', 'id_area', 'id_serv', 'id_activi', 'id_subacti', 'id_tci',
       'id_dpc', 'id_tpc', 'id_estado', 'id_orisol', 'id_cas

In [60]:
comunes = set(base1.columns).intersection(set(base2.columns)) 
base = base1[list(comunes)]
base.to_sql(name=f'{dat}', con=engine2, if_exists='append', index=False,chunksize=5000)

28167

In [61]:

query_count_after = f"SELECT COUNT(*) FROM {dat}"
cant_despues = connection2.execute(query_count_after).fetchone()[0]
print(f"Cantidad de filas en la tabla {dat} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")


Cantidad de filas en la tabla dat_cext003_essi después de la inserción: 2354310
La cantidad de filas insertadas fue de: 144167


In [62]:
#base=base.sort_values("fec_pro",ascending=False)
#fecha_fin= base.iloc[6, 1]
fecha_fin = "2024-12-31"

In [63]:
proceso = pd.read_sql_query("SELECT des_mod FROM etl_act where id_mod=3", con=connection2)
proceso = proceso.iloc[0, 0]
cod_proceso = pd.read_sql_query("SELECT id_mod FROM etl_act where id_mod=3", con=connection2)
cod_proceso = cod_proceso.iloc[0, 0]

now_fin = datetime.now()
fecha_log = now.strftime("%Y-%m-%d")
hora_log_inicio = now_inicio.strftime("%H:%M")
hora_log_fin = now_fin.strftime("%H:%M")
tabla_logs = pd.DataFrame({'esperado':control_a,'obtenido':control_d,'dim':dim,'falla':falla})
tabla_logs['proceso']=proceso
tabla_logs['dat']=dat
tabla_logs['fecha_ejecucion']=fecha_log
tabla_logs['hora_inicio']=hora_log_inicio
tabla_logs['hora_fin']=hora_log_fin
tabla_logs['faltante']=tabla_logs['esperado']-tabla_logs['obtenido']
tabla_logs['codigo']=cod_proceso
tabla_logs['fecha_ini']=fecha
tabla_logs['fecha_ter']=fecha_fin
tabla_logs['id_afectado']=id_afectado

nuevas_columnas = ['codigo', 'proceso', 'dat', 'fecha_ejecucion', 'hora_inicio','hora_fin', 'dim', 'fecha_ini','fecha_ter','esperado', 'obtenido', 'faltante','falla', 'id_afectado']

tabla_logs = tabla_logs.reindex(columns=nuevas_columnas)

In [64]:
tabla_logs

,codigo,proceso,dat,fecha_ejecucion,hora_inicio,hora_fin,dim,fecha_ini,fecha_ter,esperado,obtenido,faltante,falla,id_afectado
0,3.0,CONSULTA EXTERNA SOLICITUD ...,dat_cext003_essi,2023-06-05,11:37,11:38,dim_oricas,2023-06-01,2024-12-31,144167,144167,0,0,id_oricas
1,3.0,CONSULTA EXTERNA SOLICITUD ...,dat_cext003_essi,2023-06-05,11:37,11:38,dim_cas,2023-06-01,2024-12-31,144167,144167,0,0,id_cas
2,3.0,CONSULTA EXTERNA SOLICITUD ...,dat_cext003_essi,2023-06-05,11:37,11:38,dim_red,2023-06-01,2024-12-31,144167,144167,0,0,id_red
3,3.0,CONSULTA EXTERNA SOLICITUD ...,dat_cext003_essi,2023-06-05,11:37,11:38,dim_areas,2023-06-01,2024-12-31,144167,144167,0,0,id_area
4,3.0,CONSULTA EXTERNA SOLICITUD ...,dat_cext003_essi,2023-06-05,11:37,11:38,dim_servicios,2023-06-01,2024-12-31,144167,144167,0,0,id_serv
5,3.0,CONSULTA EXTERNA SOLICITUD ...,dat_cext003_essi,2023-06-05,11:37,11:38,dim_activi,2023-06-01,2024-12-31,144167,144167,0,0,id_activi
6,3.0,CONSULTA EXTERNA SOLICITUD ...,dat_cext003_essi,2023-06-05,11:37,11:38,dim_subacti,2023-06-01,2024-12-31,144167,144167,0,0,id_subacti
7,3.0,CONSULTA EXTERNA SOLICITUD ...,dat_cext003_essi,2023-06-05,11:37,11:38,dim_tipcit,2023-06-01,2024-12-31,144167,144167,0,0,id_tci
8,3.0,CONSULTA EXTERNA SOLICITUD ...,dat_cext003_essi,2023-06-05,11:37,11:38,dim_cexdiapresol,2023-06-01,2024-12-31,144167,144167,0,0,id_dpc
9,3.0,CONSULTA EXTERNA SOLICITUD ...,dat_cext003_essi,2023-06-05,11:37,11:38,dim_cexturprecit,2023-06-01,2024-12-31,144167,144167,0,0,id_tpc


In [65]:
tabla_logs.to_sql(name=f'logs', con=connection3, if_exists='append', index=False,chunksize=10000)

19

In [66]:
connection1.close()
connection2.close()
connection3.close()